In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/gmail_recent_1000_tagged.csv")
print(len(df))

In [ ]:
df[df["from"].str.contains("lic")].head(20)

In [ ]:
df.head(20)

In [ ]:
df['labels'].tolist()[:5]

### Labels EDA: unique values, ignore list, value counts

In [ ]:
# Labels column is stored as string (repr of a list). Parse to list of strings.
import ast

def parse_labels(s):
    """Convert 'labels' cell to list of label strings. Handles string form."""
    if pd.isna(s) or s == "":
        return []
    if isinstance(s, list):
        return s
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

df["labels_parsed"] = df["labels"].map(parse_labels)
# Show type and sample
print("Sample parsed (first row):", df["labels_parsed"].iloc[0])
print("Type of first element:", type(df["labels_parsed"].iloc[0]))

In [ ]:
# Atomic set of all label values that appear in the dataset
all_labels = set()
for lst in df["labels_parsed"]:
    all_labels.update(lst)
all_labels = sorted(all_labels)
print("All unique labels (atomic set):")
print(all_labels)

In [ ]:
len(all_labels)

In [ ]:
# Labels to ignore: system/category labels, not the ones you tagged for triage
LABELS_TO_IGNORE = [
    "INBOX", "SENT", "DRAFT", "TRASH", "SPAM", "UNREAD", "STARRED",
    "CHAT", "IMPORTANT", "YELLOW_STAR",
    "CATEGORY_PERSONAL", "CATEGORY_SOCIAL", "CATEGORY_PROMOTIONS",
    "CATEGORY_UPDATES", "CATEGORY_FORUMS",
]
# Optional: add any other system label you see in all_labels and want to ignore
print("Labels to ignore:", sorted(LABELS_TO_IGNORE))

In [ ]:
# Tag labels = the ones you care about (your triage categories)
tag_labels = sorted(set(all_labels) - set(LABELS_TO_IGNORE))
print("Tag labels (for value count):", tag_labels)

In [ ]:
# Value count: how many emails have each tag label?
def has_label(row_labels, label):
    return label in row_labels

counts = {}
for label in tag_labels:
    counts[label] = df["labels_parsed"].map(lambda lst: label in lst).sum()

vc = pd.Series(counts).sort_index()
print("Value count (emails per tag label):")
vc

In [ ]:
# Optional: value count for ignored labels (to double-check)
counts_ignore = {label: df["labels_parsed"].map(lambda lst: label in lst).sum() for label in LABELS_TO_IGNORE}
pd.Series(counts_ignore).sort_values(ascending=False)

In [ ]:
# New column: only the main label(s) you assigned (exclude LABELS_TO_IGNORE)
ignore_set = set(LABELS_TO_IGNORE)
main_label_lists = df["labels_parsed"].map(
    lambda lst: [l for l in lst if l not in ignore_set]
)
# Check how many values per row
len_counts = main_label_lists.map(len).value_counts().sort_index()
print("Number of main_label values per row:")
print(len_counts)
# Store as single string: one value -> that string; zero -> ""; multiple -> first
df["main_label"] = main_label_lists.map(
    lambda lst: lst[0] if len(lst) == 1 else (lst[0] if len(lst) > 1 else "")
)
df[["labels_parsed", "main_label"]].head()

In [ ]:
df["category"].value_counts(dropna=False)

In [ ]:
df.head()

In [ ]:
import os
os.makedirs("data", exist_ok=True)
out_path = "data/gmail_tagged_emails.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")

In [ ]:
df.columns

In [ ]:
df["main_label"].value_counts()

In [ ]:
df1 = pd.read_csv("data/gmail_tagged_sample.csv")
df2 = pd.read_csv("data/gmail_tagged_sample_updated.csv")

print(len(df1),len(df2))

In [ ]:
df1.head()

In [ ]:
df1 = df1.rename(columns={"main_label":"label_old"})
df2 = df2.rename(columns={"main_label":"label_new"})

In [ ]:
df_cmp = df1.merge(df2,on="id",how="inner")
print(len(df_cmp))

In [ ]:
print(df_cmp["label_old"].nunique())
print(df_cmp["label_new"].nunique())

In [ ]:
df_mismatch = df_cmp[df_cmp["label_new"]!=df_cmp["label_old"]]
print(len(df_mismatch))

In [ ]:
df_mismatch